In [1]:
import zmq
import json
import pandas as pd
import numpy as np
from datetime import timedelta
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import signal
import time
import sys
import atexit; atexit.register(lambda: print("ATExit cleanup!"))
import matplotlib.pyplot as plt


In [2]:
def create_features(df):
    """
    Create time series features and lag features based on time series index.
    """
    df = df.copy()

    # Basic time-based features
    df['minute'] = df.index.minute
    df['hour'] = df.index.hour
    df['day'] = df.index.day
    df['dayofweek'] = df.index.dayofweek
    df['month'] = df.index.month

    # Lag features
    df['lag_1minute'] = df['Value'].shift(1)  # 1 minute lag
    df['lag_1h'] = df['Value'].shift(60)   # 1 hour lag
    df['lag_1d'] = df['Value'].shift(1440)  # 1 day lag

    # Rolling statistics features
    df['rolling_mean_30minutes'] = df['Value'].rolling(window=30).mean()  # Last 30 minutes rolling mean
    df['rolling_mean_3hours'] = df['Value'].rolling(window=180).mean()  # Last 3 hours rolling mean
    df['rolling_mean_1days'] = df['Value'].rolling(window=1440).mean()  # Last 1 day rolling mean
    df['rolling_mean_same_hour_last_day'] = df['Value'].shift(1440).rolling(window=30).mean()  # Same hour previous day rolling mean

    return df

In [ ]:
def prepare_features(df):
    """Prepare advanced features matching our app_model_xgb."""
    df = df.copy()
    df = create_features(df)  # Using our existing create_features
    return df.dropna()

In [4]:
def signal_handler(sig, frame):
    print("\nReceived signal to stop subscriber.")
    sys.exit(0)
    
# Register signal handlers for clean shutdown
signal.signal(signal.SIGINT, signal_handler)
signal.signal(signal.SIGTERM, signal_handler)

<Handlers.SIG_DFL: 0>

In [5]:
# Set up ZeroMQ subscriber
context = zmq.Context()
socket = context.socket(zmq.SUB)
socket.connect("tcp://localhost:5556")
socket.setsockopt_string(zmq.SUBSCRIBE, "")  # Subscribe to all messages


In [6]:
# Initialize variables
data = []
model = None
last_train_date = None
test_actuals = []
test_predictions = []
flags = []
seen_dates = set()
train_times = []
total_flags=0

TARGET = 'Value'
FEATURES_XGB = [
    'hour', 'dayofweek', 'month', 'minute', 'day', 'lag_1minute', 'lag_1h', 'lag_1d',
    'rolling_mean_30minutes', 'rolling_mean_3hours', 'rolling_mean_1days', 'rolling_mean_same_hour_last_day'
]

In [ ]:
def train_model(X, y, model):
    """
    Train XGBoost using the exact configuration from app_model_xgb.
    Assumes X is a DataFrame with FEATURES_XGB columns, y is the target Series.
    """
    
    # Using only the specified features
    X = X[FEATURES_XGB]
    
    # Creating DMatrix for XGBoost
    dtrain = xgb.DMatrix(X, label=y)
    
    # Setting parameters for XGBoost
    params = {
        'objective': 'reg:squarederror',  # Objective function for regression
        'eval_metric': 'rmse',  # Evaluation metric
        'max_depth': 3,  # Depth of the trees
        'learning_rate': 0.01,  # Learning rate
        'colsample_bytree': 0.8,  # Subsample of features
        'subsample': 0.8  # Subsample ratio
    }

    # Number of boosting rounds and early stopping
    num_round = 1000  # Number of boosting rounds
    if model is None:
        # Training the empty model (cold start)
        reg = xgb.train(params, dtrain, num_round, verbose_eval=100)  
    else:
        # Continue training from existing model (warm start)
        incremental_rounds=300
        reg = xgb.train(
            params, 
            dtrain, 
            num_round,
            xgb_model=model,  # Warm start from previous model
            verbose_eval=100
        )
    return reg  # Returns the Booster object for prediction

In [8]:
print("Subscriber starting... Waiting for heart rate data from publisher.")
print("Will train on after first 2 days, predict/flag on next day, retrain, repeat everyday.")
print("Stop with VSCode button or Ctrl+C—cleanup will run.")

Subscriber starting... Waiting for heart rate data from publisher.
Will train on after first 2 days, predict/flag on next day, retrain, repeat everyday.
Stop with VSCode button or Ctrl+C—cleanup will run.


In [ ]:
try:
    while True:
        message = socket.recv_string()
        msg = json.loads(message)
        dt = pd.to_datetime(msg['datetime'])
        hr = msg['hr']
        

        # Append new data point
        data.append({'Time': dt, 'Value': hr})
        temp_df = pd.DataFrame(data).set_index('Time').sort_index()
        current_date = dt.date()

        # Track new days
        if current_date not in seen_dates:
            seen_dates.add(current_date)

            # Initial training: on first 2 days (triggered on first msg of day 3)
            if model is None and len(seen_dates) == 3:
                train_end = current_date - timedelta(days=1)
                train_df = temp_df[temp_df.index.date <= train_end]
                train_features = prepare_features(train_df)
                X_train = train_features[FEATURES_XGB]
                y_train = train_df.loc[X_train.index, TARGET]
                start_train = time.time()
                model = train_model(X_train, y_train, model)
                elapsed = time.time() - start_train
                train_times.append(elapsed)
                last_train_date = train_end
                print(f"Initial model trained on data up to {train_end} (2 days). Training time: {elapsed:.2f} sec")
                test_actuals, test_predictions, flags = [], [], []

            # Daily retraining: after initial, on start of each new day (simulates \"every night after 12:00\")
            elif model is not None and current_date > last_train_date:
                # Computing and printing metrics for previous test day
                if test_actuals:
                    rmse = np.sqrt(mean_squared_error(test_actuals, test_predictions))
                    mae = mean_absolute_error(test_actuals, test_predictions)
                    print(f"\nPrevious Test Period - RMSE: {rmse:.4f}, MAE: {mae:.4f}")
                    print(f"Anomaly Flags (1): {sum(flags)} / {len(flags)}")
                    test_actuals, test_predictions, flags = [], [], []

                # Retrain on only the data for yesterday using prepared features computed from full history
                new_train_end = current_date - timedelta(days=1)
                train_df_all = temp_df[temp_df.index.date <= new_train_end]
                train_features_all = prepare_features(train_df_all)
                X_all = train_features_all[FEATURES_XGB] # All features up to new_train_end
                X_train = X_all[X_all.index.date == new_train_end] # Features only for new_train_end day
                y_train = train_df_all.loc[X_train.index, TARGET] if len(X_train) > 0 else None

                if X_train is None or len(X_train) == 0:
                    print(f"No data for retrain date {new_train_end}; skipping retrain.")
                else:
                    start_train = time.time() # start timing for retrain
                    model = train_model(X_train, y_train, model)
                    elapsed = time.time() - start_train # end timing for retrain
                    train_times.append(elapsed)
                    last_train_date = new_train_end
                    print(f"Model retrained on data for {new_train_end}. Training time: {elapsed:.4f} sec")

        # Predict if model is trained and this is a test point
        if model is not None and dt.date() > last_train_date and len(temp_df)> 1:
            # Re-prepare features for the full temp_df to compute rollings/lags
            full_features = prepare_features(temp_df)
            
            # Get the last row for prediction (dropna ensures we have features)
            if len(full_features) > 0:
                last_row = full_features.iloc[-1:]
                X_pred = last_row[FEATURES_XGB]
                dpred = xgb.DMatrix(X_pred)
                pred = model.predict(dpred)[0]
                
                # Compute relative deviation and flag
                rel_dev = abs(pred - hr) / hr if hr != 0 else 0
                #flag = 1 if rel_dev >= 0.20 else 0

                # Enhanced flag: 1 if >20% deviation and outside resting range (60-100 bpm)
                hr_outside_range = hr < 60 or hr > 100
                flag = 1 if (rel_dev >= 0.20 and hr_outside_range) else 0
                if flag:
                    total_flags=total_flags+1
                    # Print real-time results (flush to ensure immediate output when anomaly detected)
                    print(f"Anomaly detected at {dt.strftime('%Y-%m-%d %H:%M')}: Actual={hr:.1f}, Predicted={pred:.1f}, RelDev={rel_dev*100:.1f}%, Total Flag={total_flags}" )
                
                # Collect for period metrics
                test_actuals.append(hr)
                test_predictions.append(pred)
                flags.append(flag)
except KeyboardInterrupt:
    print("\nSubscriber stopped by user (KeyboardInterrupt).")
    # Final metrics if any pending
    if test_actuals:
        rmse = np.sqrt(mean_squared_error(test_actuals, test_predictions))
        mae = mean_absolute_error(test_actuals, test_predictions)
        print(f"\nFinal Partial Test - RMSE: {rmse:.4f}, MAE: {mae:.4f}")
        print(f"Final Flags (1): {sum(flags)} / {len(flags)}")

except Exception as e:
    print(f"\nSubscriber stopped due to error: {e}")
    # Still run final metrics
    if test_actuals:
        rmse = np.sqrt(mean_squared_error(test_actuals, test_predictions))
        mae = mean_absolute_error(test_actuals, test_predictions)
        print(f"\nFinal Partial Test - RMSE: {rmse:.4f}, MAE: {mae:.4f}")
        print(f"Final Flags (1): {sum(flags)} / {len(flags)}")

finally:
    print("Cleaning up... Closing socket and context.", flush=True)
    # Final metrics one last time (in case except didn't run)
    if test_actuals:
        rmse = np.sqrt(mean_squared_error(test_actuals, test_predictions))
        mae = mean_absolute_error(test_actuals, test_predictions)
        print(f"\nFinal Cleanup Metrics - RMSE: {rmse:.4f}, MAE: {mae:.4f}")
        print(f"Final Flags (1): {sum(flags)} / {len(flags)}")
    socket.close()
    context.term()
    print("Subscriber closed successfully.", flush=True)
